# 03 — Final input and split setup

This notebook validates the single 26-feature primary input and freezes every patient assignment used by the final analysis.

For each seed from 0 through 19:

- 70% of patients form the **outer training set** used for all development;
- 30% form the **outer test set**, scored once after the pipeline is selected;
- the outer training set is divided into five **inner validation folds** for statistical analysis, preprocessing choices, feature selection, model-family screening, and tuning.

Direct and two-stage classification reuse the same assignments. Stage 2 later restricts each applicable training partition to true rare and recurrent fallers.

This notebook performs no imputation, feature engineering, feature selection, or model fitting.


## 1. Set up the final-run paths

Load only standard analysis libraries and locate the project root explicitly so the notebook runs from any folder inside the repository.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, train_test_split

working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

print("Project root:", project_root)


Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk


Declare the Notebook 02 input directory and a new output directory for the frozen manifests.


In [2]:
aggregation_directory = (
    project_root
    / "results/final_pipeline/06_final_fit_and_performance_summary/02_patient_level_aggregation"
)
output_directory = (
    project_root
    / "results/final_pipeline/06_final_fit_and_performance_summary/03_setup_and_splits"
)
output_directory.mkdir(parents=True, exist_ok=True)

predictor_path = aggregation_directory / "primary_1040_26_predictors.csv"
metadata_path = aggregation_directory / "primary_1040_outcome_metadata.csv"

required_inputs = [predictor_path, metadata_path]
missing_inputs = [path.name for path in required_inputs if not path.is_file()]
assert not missing_inputs, f"Missing Notebook 02 inputs: {missing_inputs}"

print("Inputs:", aggregation_directory.relative_to(project_root))
print("Outputs:", output_directory.relative_to(project_root))


Inputs: results/final_pipeline/06_final_fit_and_performance_summary/02_patient_level_aggregation
Outputs: results/final_pipeline/06_final_fit_and_performance_summary/03_setup_and_splits


## 2. Load and validate the final primary input

Read the unimputed predictor table and its separate outcome metadata. Patient numbers are treated as identifiers rather than numeric measurements.


In [3]:
predictors = pd.read_csv(
    predictor_path,
    dtype={"PATNO": "string"},
    low_memory=False,
)
metadata = pd.read_csv(
    metadata_path,
    dtype={"PATNO": "string"},
    parse_dates=["index_date", "outcome_date"],
)

assert set(predictors["PATNO"]) == set(metadata["PATNO"])

predictors = (
    predictors.assign(_patient_order=pd.to_numeric(predictors["PATNO"], errors="raise"))
    .sort_values("_patient_order")
    .drop(columns="_patient_order")
    .reset_index(drop=True)
)
metadata = (
    metadata.assign(_patient_order=pd.to_numeric(metadata["PATNO"], errors="raise"))
    .sort_values("_patient_order")
    .drop(columns="_patient_order")
    .reset_index(drop=True)
)

assert predictors["PATNO"].equals(metadata["PATNO"])
print("Predictor shape:", predictors.shape)
print("Metadata shape:", metadata.shape)


Predictor shape: (1040, 27)
Metadata shape: (1040, 5)


Declare the exact 26-feature contract. This prevents an accidental column addition, omission, or reordering from silently changing the final run.


In [4]:
primary_features = [
    "Years_since_PD_diagnosis", "Age", "DXPOSINS", "DXRIGID", "DOPTHERST",
    "MCATOT", "FRZGT12M", "SCAU14", "SCAU16", "GDS_TOTAL",
    "NQ_GAUSSIAN_REVISION", "NP1RTOT", "BMI", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "FEATPOSHYP", "NP3TOT_COMBINED_MAX", "NP4TOT", "ANYFAMPD", "DXTREMOR",
    "DXBRADY", "DOMSIDE", "NP1CNST",
]
nominal_features = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}

assert predictors.columns.tolist() == ["PATNO", *primary_features]
assert len(primary_features) == len(set(primary_features)) == 26

feature_manifest = pd.DataFrame({
    "position": range(1, len(primary_features) + 1),
    "feature": primary_features,
    "kind": [
        "nominal" if feature in nominal_features else "numeric_or_ordinal"
        for feature in primary_features
    ],
    "missing_patients": [predictors[feature].isna().sum() for feature in primary_features],
})
feature_manifest["missing_percent"] = (
    100 * feature_manifest["missing_patients"] / len(predictors)
).round(1)


Confirm the cohort, outcome classes, 12-month landmark lag, and absence of outcome fields from the predictor table. Missing predictor values remain untouched for fold-fitted preprocessing.


In [5]:
expected_class_counts = {0: 712, 1: 227, 2: 101}
observed_class_counts = (
    metadata["falls_class"].value_counts().sort_index().to_dict()
)
expected_outcome_dates = metadata["index_date"] + pd.DateOffset(months=12)
forbidden_predictors = {"falls_raw", "falls_class", "index_date", "outcome_date"}

assert len(predictors) == len(metadata) == 1_040
assert predictors["PATNO"].is_unique and metadata["PATNO"].is_unique
assert observed_class_counts == expected_class_counts
assert expected_outcome_dates.equals(metadata["outcome_date"])
assert forbidden_predictors.isdisjoint(predictors.columns)
assert predictors[primary_features].isna().any().any()

class_distribution = pd.DataFrame({
    "falls_class": [0, 1, 2],
    "label": ["no fall", "rare fall", "recurrent fall"],
    "patients": [expected_class_counts[class_id] for class_id in [0, 1, 2]],
})
class_distribution["percent"] = (
    100 * class_distribution["patients"] / len(metadata)
).round(1)

display(class_distribution)
display(feature_manifest)


,falls_class,label,patients,percent
0,0,no fall,712,68.5
1,1,rare fall,227,21.8
2,2,recurrent fall,101,9.7


,position,feature,kind,missing_patients,missing_percent
0,1,Years_since_PD_diagnosis,numeric_or_ordinal,0,0.0
1,2,Age,numeric_or_ordinal,0,0.0
2,3,DXPOSINS,nominal,1,0.1
3,4,DXRIGID,nominal,0,0.0
4,5,DOPTHERST,nominal,202,19.4
5,6,MCATOT,numeric_or_ordinal,0,0.0
6,7,FRZGT12M,numeric_or_ordinal,63,6.1
7,8,SCAU14,numeric_or_ordinal,0,0.0
8,9,SCAU16,numeric_or_ordinal,0,0.0
9,10,GDS_TOTAL,numeric_or_ordinal,0,0.0


## 3. Freeze the evaluation configuration

One **outer split** is one complete experiment for a seed. All pipeline decisions use only its 728 training patients. Its 312 test patients provide the final score for that seed.

Each outer training set is divided into five folds. Four folds train a candidate and the fifth validates it; this rotates until every outer-training patient has served once in inner validation.


In [6]:
SPLIT_SEEDS = list(range(20))
TEST_FRACTION = 0.30
N_INNER_FOLDS = 5
PRIMARY_METRIC = "macro_f1"
NEAR_TIE_MARGIN = 0.01

configuration = pd.DataFrame([
    {"parameter": "primary_patients", "value": len(metadata)},
    {"parameter": "candidate_features", "value": len(primary_features)},
    {"parameter": "outer_method", "value": "stratified 70/30 holdout"},
    {"parameter": "outer_splits", "value": len(SPLIT_SEEDS)},
    {"parameter": "split_seeds", "value": ",".join(map(str, SPLIT_SEEDS))},
    {"parameter": "test_fraction", "value": TEST_FRACTION},
    {"parameter": "inner_method", "value": "stratified k-fold within outer training"},
    {"parameter": "inner_folds", "value": N_INNER_FOLDS},
    {"parameter": "inner_seed_rule", "value": "same integer as split_seed"},
    {"parameter": "primary_selection_metric", "value": PRIMARY_METRIC},
    {"parameter": "near_tie_margin", "value": NEAR_TIE_MARGIN},
    {"parameter": "architectures", "value": "direct|two_stage"},
    {"parameter": "outer_test_use", "value": "score once after training-only selection"},
])

display(configuration)


,parameter,value
0,primary_patients,1040
1,candidate_features,26
2,outer_method,stratified 70/30 holdout
3,outer_splits,20
4,split_seeds,"0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19"
5,test_fraction,0.3
6,inner_method,stratified k-fold within outer training
7,inner_folds,5
8,inner_seed_rule,same integer as split_seed
9,primary_selection_metric,macro_f1


Record the three target definitions and their approved near-tie recalls. These targets share the same saved patient partitions.


In [7]:
target_manifest = pd.DataFrame([
    {
        "target": "direct",
        "eligible_training_patients": "all outer-training patients",
        "label_definition": "falls_class: 0=no, 1=rare, 2=recurrent",
        "near_tie_recall": "rare-fall recall",
    },
    {
        "target": "stage_1",
        "eligible_training_patients": "all outer-training patients",
        "label_definition": "0=no fall, 1=any fall",
        "near_tie_recall": "any-fall recall",
    },
    {
        "target": "stage_2",
        "eligible_training_patients": "true rare and recurrent fallers only",
        "label_definition": "0=rare fall, 1=recurrent fall",
        "near_tie_recall": "rare-fall recall",
    },
])

display(target_manifest)


,target,eligible_training_patients,label_definition,near_tie_recall
0,direct,all outer-training patients,"falls_class: 0=no, 1=rare, 2=recurrent",rare-fall recall
1,stage_1,all outer-training patients,"0=no fall, 1=any fall",any-fall recall
2,stage_2,true rare and recurrent fallers only,"0=rare fall, 1=recurrent fall",rare-fall recall


## 4. Create the 20 paired outer splits

Apply the revision-style stratified 70/30 split for seeds 0–19. Numeric patient-number ordering is fixed before splitting so the assignments are portable and reproducible.


In [8]:
patient_indices = np.arange(len(metadata))
y = metadata["falls_class"].to_numpy()
class_labels = {0: "no fall", 1: "rare fall", 2: "recurrent fall"}

outer_rows = []
outer_balance_rows = []
outer_train_indices = {}

for split_seed in SPLIT_SEEDS:
    train_indices, test_indices = train_test_split(
        patient_indices,
        test_size=TEST_FRACTION,
        stratify=y,
        random_state=split_seed,
    )
    outer_train_indices[split_seed] = train_indices

    for role, indices in [("train", train_indices), ("test", test_indices)]:
        outer_rows.extend({
            "split_seed": split_seed,
            "PATNO": metadata.at[index, "PATNO"],
            "role": role,
        } for index in indices)

        counts = metadata.iloc[indices]["falls_class"].value_counts()
        for fall_class in [0, 1, 2]:
            patients = int(counts.get(fall_class, 0))
            outer_balance_rows.append({
                "split_seed": split_seed,
                "role": role,
                "falls_class": fall_class,
                "label": class_labels[fall_class],
                "patients": patients,
                "percent_within_role": round(100 * patients / len(indices), 1),
            })

outer_assignments = pd.DataFrame(outer_rows)
outer_assignments["_role_order"] = outer_assignments["role"].map({"train": 0, "test": 1})
outer_assignments = (
    outer_assignments
    .sort_values(["split_seed", "_role_order", "PATNO"])
    .drop(columns="_role_order")
    .reset_index(drop=True)
)
outer_class_balance = pd.DataFrame(outer_balance_rows)


Show only split sizes and class balance. Patient identifiers remain in the saved assignment file and are not displayed.


In [9]:
outer_size_summary = (
    outer_assignments.groupby(["split_seed", "role"])
    .size()
    .unstack("role")
)
outer_balance_summary = (
    outer_class_balance.groupby(["role", "falls_class"])["patients"]
    .agg(["min", "max"])
    .reset_index()
)

display(outer_size_summary.head())
display(outer_balance_summary)


role,test,train
split_seed,,
0,312,728
1,312,728
2,312,728
3,312,728
4,312,728


,role,falls_class,min,max
0,test,0,214,214
1,test,1,68,68
2,test,2,30,30
3,train,0,498,498
4,train,1,159,159
5,train,2,71,71


## 5. Create five inner folds within every outer training set

Assign each outer-training patient to exactly one inner validation fold. The remaining four folds become that iteration's inner training data. Outer-test patients never receive an inner-fold assignment.


In [10]:
inner_rows = []
inner_balance_rows = []

for split_seed in SPLIT_SEEDS:
    train_indices = outer_train_indices[split_seed]
    train_labels = y[train_indices]
    inner_cv = StratifiedKFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=split_seed,
    )

    for inner_fold, (inner_train_positions, validation_positions) in enumerate(
        inner_cv.split(train_indices, train_labels)
    ):
        inner_train_indices = train_indices[inner_train_positions]
        validation_indices = train_indices[validation_positions]

        inner_rows.extend({
            "split_seed": split_seed,
            "inner_seed": split_seed,
            "PATNO": metadata.at[index, "PATNO"],
            "inner_validation_fold": inner_fold,
        } for index in validation_indices)

        for role, indices in [
            ("inner_train", inner_train_indices),
            ("inner_validation", validation_indices),
        ]:
            counts = metadata.iloc[indices]["falls_class"].value_counts()
            for fall_class in [0, 1, 2]:
                patients = int(counts.get(fall_class, 0))
                inner_balance_rows.append({
                    "split_seed": split_seed,
                    "inner_seed": split_seed,
                    "inner_validation_fold": inner_fold,
                    "role": role,
                    "falls_class": fall_class,
                    "label": class_labels[fall_class],
                    "patients": patients,
                    "percent_within_role": round(100 * patients / len(indices), 1),
                })

inner_assignments = (
    pd.DataFrame(inner_rows)
    .sort_values(["split_seed", "inner_validation_fold", "PATNO"])
    .reset_index(drop=True)
)
inner_class_balance = pd.DataFrame(inner_balance_rows)


Summarize the inner-fold sizes and the smallest class count. These counts show that direct classification and both binary stages remain estimable in every fold.


In [11]:
inner_size_summary = (
    inner_assignments.groupby(["split_seed", "inner_validation_fold"])
    .size()
    .rename("validation_patients")
    .reset_index()
)
inner_balance_summary = (
    inner_class_balance.groupby(["role", "falls_class"])["patients"]
    .agg(["min", "max"])
    .reset_index()
)

display(inner_size_summary.groupby("split_seed")["validation_patients"].agg(["min", "max"]).head())
display(inner_balance_summary)


,min,max
split_seed,,
0,145,146
1,145,146
2,145,146
3,145,146
4,145,146


,role,falls_class,min,max
0,inner_train,0,398,399
1,inner_train,1,127,128
2,inner_train,2,56,57
3,inner_validation,0,99,100
4,inner_validation,1,31,32
5,inner_validation,2,14,15


## 6. Validate separation, pairing, and class coverage

Collect every structural check in one table. A failed check blocks saving the split manifests.


In [12]:
validation_rows = []


def check(name, condition, detail):
    validation_rows.append({
        "check": name,
        "passed": bool(condition),
        "detail": detail,
    })


Validate the primary input contract and confirm that all 20 outer test sets are distinct.


In [13]:
check("primary patient count", len(metadata) == 1_040, "expected 1,040")
check("primary feature count", len(primary_features) == 26, "expected 26")
check(
    "predictor and metadata alignment",
    predictors["PATNO"].equals(metadata["PATNO"]),
    "same patients in deterministic order",
)
check(
    "primary class counts",
    observed_class_counts == expected_class_counts,
    "expected 712/227/101",
)
check(
    "12-month landmark lag",
    expected_outcome_dates.equals(metadata["outcome_date"]),
    "every outcome month is 12 months after its index month",
)
check(
    "outcome separation",
    forbidden_predictors.isdisjoint(predictors.columns),
    "predictors contain no outcome or timing fields",
)
check(
    "unimputed predictors",
    predictors[primary_features].isna().any().any(),
    "missing values remain for training-only handling",
)

test_signatures = (
    outer_assignments.loc[outer_assignments["role"].eq("test")]
    .groupby("split_seed")["PATNO"]
    .apply(lambda values: tuple(sorted(values)))
)
check(
    "distinct outer test sets",
    len(set(test_signatures)) == len(SPLIT_SEEDS),
    "20 unique patient sets",
)


Validate every outer and inner partition without printing identifiers. Inner-fold checks also require both faller classes for the later Stage 2 analysis.


In [14]:
patient_ids = set(metadata["PATNO"])
outer_separation_ok = True
outer_sizes_ok = True
inner_coverage_ok = True
inner_separation_ok = True
stage_2_coverage_ok = True

for split_seed in SPLIT_SEEDS:
    seed_outer = outer_assignments.loc[
        outer_assignments["split_seed"].eq(split_seed)
    ]
    train_ids = set(seed_outer.loc[seed_outer["role"].eq("train"), "PATNO"])
    test_ids = set(seed_outer.loc[seed_outer["role"].eq("test"), "PATNO"])

    outer_separation_ok &= train_ids.isdisjoint(test_ids)
    outer_separation_ok &= train_ids | test_ids == patient_ids
    outer_sizes_ok &= len(train_ids) == 728 and len(test_ids) == 312

    seed_inner = inner_assignments.loc[
        inner_assignments["split_seed"].eq(split_seed)
    ]
    inner_coverage_ok &= set(seed_inner["PATNO"]) == train_ids
    inner_coverage_ok &= seed_inner["PATNO"].value_counts().eq(1).all()
    inner_coverage_ok &= seed_inner["inner_validation_fold"].nunique() == 5
    inner_separation_ok &= test_ids.isdisjoint(set(seed_inner["PATNO"]))

    seed_stage_2 = inner_class_balance.loc[
        inner_class_balance["split_seed"].eq(split_seed)
        & inner_class_balance["falls_class"].isin([1, 2])
    ]
    stage_2_coverage_ok &= seed_stage_2["patients"].gt(0).all()

check("outer split separation", outer_separation_ok, "train and test are disjoint and exhaustive")
check("outer split sizes", outer_sizes_ok, "728 train and 312 test for every seed")
check("inner assignment coverage", inner_coverage_ok, "each outer-training patient enters one validation fold")
check("outer-test exclusion from inner folds", inner_separation_ok, "zero patient overlap")
check(
    "all classes in outer roles",
    outer_class_balance["patients"].gt(0).all(),
    "classes 0, 1, and 2 occur in every train and test set",
)
check(
    "all classes in inner roles",
    inner_class_balance["patients"].gt(0).all(),
    "classes 0, 1, and 2 occur in every inner train and validation set",
)
check(
    "Stage 2 class coverage",
    stage_2_coverage_ok,
    "rare and recurrent fallers occur in every inner role",
)


Display the concise validation result and stop if any requirement failed.


In [15]:
validation = pd.DataFrame(validation_rows)
display(validation)

assert validation["passed"].all(), validation.loc[~validation["passed"]]
print(f"Validation checks passed: {validation['passed'].sum()}/{len(validation)}")


,check,passed,detail
0,primary patient count,True,"expected 1,040"
1,primary feature count,True,expected 26
2,predictor and metadata alignment,True,same patients in deterministic order
3,primary class counts,True,expected 712/227/101
4,12-month landmark lag,True,every outcome month is 12 months after its ind...
5,outcome separation,True,predictors contain no outcome or timing fields
6,unimputed predictors,True,missing values remain for training-only handling
7,distinct outer test sets,True,20 unique patient sets
8,outer split separation,True,train and test are disjoint and exhaustive
9,outer split sizes,True,728 train and 312 test for every seed


Validation checks passed: 15/15


## 7. Save the frozen manifests

Write each artifact only when it is new or exactly matches an existing file. A differing existing file raises an error instead of being overwritten silently.


In [16]:
def save_new_or_identical(frame, path):
    csv_text = frame.to_csv(index=False)
    if path.is_file():
        if path.read_text() != csv_text:
            raise FileExistsError(
                f"Existing artifact differs and was not overwritten: {path.name}"
            )
        return "already identical"

    path.write_text(csv_text)
    return "created"


artifacts = {
    "primary_feature_manifest.csv": feature_manifest,
    "primary_class_distribution.csv": class_distribution,
    "analysis_configuration.csv": configuration,
    "target_manifest.csv": target_manifest,
    "outer_split_assignments.csv": outer_assignments,
    "outer_class_balance.csv": outer_class_balance,
    "inner_fold_assignments.csv": inner_assignments,
    "inner_class_balance.csv": inner_class_balance,
    "split_validation.csv": validation,
}

save_rows = []
for filename, frame in artifacts.items():
    status = save_new_or_identical(frame, output_directory / filename)
    save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

save_summary = pd.DataFrame(save_rows)
display(save_summary)
print("No predictor values were transformed and no model was fitted.")


,artifact,status,rows
0,primary_feature_manifest.csv,created,26
1,primary_class_distribution.csv,created,3
2,analysis_configuration.csv,created,13
3,target_manifest.csv,created,3
4,outer_split_assignments.csv,created,20800
5,outer_class_balance.csv,created,120
6,inner_fold_assignments.csv,created,14560
7,inner_class_balance.csv,created,600
8,split_validation.csv,created,15


No predictor values were transformed and no model was fitted.


## Result required before Notebook 04

All validation checks must pass. The saved assignments then become the only split definitions used by the statistical analysis, feature-pipeline screening, model-family screening, tuning, outer evaluation, and same-cohort sensitivities.

Notebook 04 may calculate outcome-group statistics only inside the applicable saved training partitions. It must never use an outer test partition to choose a feature.
